In [ ]:
import os

import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

/Users/admin/anaconda3/envs/qolda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
token = os.getenv("HF_TOKEN")

In [4]:
DATASET = "issai/kazsandra"
CONFIG = "polarity_classification"
LABELS = {0: "negative", 1: "positive"}

dataset_train = load_dataset(DATASET, CONFIG, split="train", token=token)
dataset_test = load_dataset(DATASET, CONFIG, split="test", token=token)

df_train = dataset_train.to_pandas()
df_test = dataset_test.to_pandas()

In [5]:
df_train.head()

,custom_id,text,text_cleaned,label,domain
0,pla015439pla,Өтте күшті,өтте күшті,1,appstore
1,pla083193pla,Мәбазар жок .Оте керемет тамаша керемет,мәбазар жок оте керемет тамаша керемет,1,appstore
2,pla113624pla,"Кушти , дал тура айтады 👍👍👍👍",кушти дал тура айтады,1,appstore
3,pla029825pla,Реклама коп,реклама коп,0,appstore
4,pla002604pla,5-баға беремін,5 баға беремін,1,appstore


In [6]:
df_test.head()

,custom_id,text,text_cleaned,label,domain
0,pla096342pla,тез тарту керек 5 минутта тартылатын болсын.,тез тарту керек 5 минутта тартылатын болсын,1,appstore
1,pla026147pla,Каспи рахмет сендерге куптен куттим кашан прил...,каспи рахмет сендерге куптен куттим кашан прил...,1,appstore
2,pla124923pla,Маған ұнады өте керемет телефоным как будто ай...,маған ұнады өте керемет телефоным как будто ай...,1,appstore
3,pla024316pla,Өте іңғайлы үнады маған,өте іңғайлы үнады маған,1,appstore
4,pla104101pla,"Ассалаумағалейкум, ОҚЫП ЖАУАБЫН БЕРСЕҢІЗДЕР ЕК...",ассалаумағалейкум оқып жауабын берсеңіздер еке...,1,appstore


In [11]:
def explore(df, name):
    print(f"\n{name}: {len(df)} reviews")

    counts = df["label"].value_counts().sort_index()
    for label, n in counts.items():
        print(f"  label {label} ({LABELS[label]:>8}): {n:>6}  {100 * n / len(df):5.1f}%")

    print("  domains:", df["domain"].value_counts().to_dict())

    words = df["text_cleaned"].astype(str).str.split().str.len()
    print(f"  words per review: median={words.median():.0f}  mean={words.mean():.1f}  max={words.max()}")
    print(f"  very short (<= 3 words): {100 * (words <= 3).mean():.1f}%")

    for label in counts.index:
        example = df.loc[df["label"] == label, "text"].iloc[0]
        print(f"  label {label} ({LABELS[label]}): «{str(example)[:60]}»")
        
explore(df_train, "train")


train: 134368 reviews
  label 0 (negative):  23951   17.8%
  label 1 (positive): 110417   82.2%
  domains: {'appstore': 101477, 'market': 22561, 'mapping': 6509, 'bookstore': 3821}
  words per review: median=6  mean=8.7  max=271
  very short (<= 3 words): 22.8%
  label 0 (negative): «Реклама коп»
  label 1 (positive): «Өтте күшті»


In [12]:
explore(df_test, "test")


test: 16797 reviews
  label 0 (negative):   2993   17.8%
  label 1 (positive):  13804   82.2%
  domains: {'appstore': 12685, 'market': 2820, 'mapping': 814, 'bookstore': 478}
  words per review: median=6  mean=8.6  max=137
  very short (<= 3 words): 22.5%
  label 0 (negative): «Рахатты гдзыдан аламыз»
  label 1 (positive): «тез тарту керек 5 минутта тартылатын болсын.»


In [14]:
def balance(df, per_class, seed=42):
    """Take the same number of reviews from each class."""
    parts = [group.sample(per_class, random_state=seed)
             for _, group in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

TRAIN_SIZE = 4000
TEST_SIZE = 1000

train = balance(df_train, TRAIN_SIZE)
test = balance(df_test, TEST_SIZE)

In [16]:
explore(train, "train")


train: 8000 reviews
  label 0 (negative):   4000   50.0%
  label 1 (positive):   4000   50.0%
  domains: {'appstore': 6414, 'market': 983, 'mapping': 365, 'bookstore': 238}
  words per review: median=6  mean=9.1  max=122
  very short (<= 3 words): 24.9%
  label 0 (negative): «Маған ұнаган жоқ коляска, дөңгелектеры өте жаман нашар айнал»
  label 1 (positive): «Өте ыңғайлы шаңсорғыш.»


In [17]:
explore(test, "test")


test: 2000 reviews
  label 0 (negative):   1000   50.0%
  label 1 (positive):   1000   50.0%
  domains: {'appstore': 1631, 'market': 219, 'mapping': 101, 'bookstore': 49}
  words per review: median=6  mean=8.9  max=97
  very short (<= 3 words): 24.4%
  label 0 (negative): «Жасамайды реклама көп 0% 👎»
  label 1 (positive): «Бұл өте керемет придложения ,бұл маған көп көмектесті»


In [18]:
def build(name):
    """TF-IDF features plus one classifier, as a single Pipeline."""
    classifiers = {
        "nb": MultinomialNB(alpha=1.0),
        "logreg": LogisticRegression(max_iter=1000, C=5.0),
        "svm": LinearSVC(C=1.0),
        # Trees split on one feature at a time, which is a poor fit for tens of
        # thousands of sparse columns. They are here to be measured, not assumed.
        "tree": DecisionTreeClassifier(max_depth=40, random_state=42),
        "forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    }
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", classifiers[name]),
    ])

In [22]:
def evaluate(model, x_test, y_test, name):
    predicted = model.predict(x_test)
    print(f"\n--- {name} ---")
    print(classification_report(y_test, predicted,
                                target_names=[LABELS[0], LABELS[1]], digits=3))
    print("confusion matrix (rows = true, columns = predicted)")
    print(confusion_matrix(y_test, predicted))
    return predicted

In [19]:
x_train, y_train = train["text_cleaned"].astype(str), train["label"]
x_test, y_test = test["text_cleaned"].astype(str), test["label"]

### Training a model

In [24]:
name = "nb"

model = build(name)
model.fit(x_train, y_train)
print(f"  features: {len(model.named_steps['tfidf'].get_feature_names_out())}")
print(f"  train accuracy: {model.score(x_train, y_train):.3f}")
print(f"  test  accuracy: {model.score(x_test, y_test):.3f}")
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"  5-fold CV: {scores.mean():.3f} ± {scores.std():.3f}  {scores.round(3)}")
predicted = evaluate(model, x_test, y_test, name)

  features: 9419
  train accuracy: 0.867
  test  accuracy: 0.784
  5-fold CV: 0.792 ± 0.006  [0.789 0.788 0.785 0.797 0.801]

--- nb ---
              precision    recall  f1-score   support

    negative      0.782     0.787     0.785      1000
    positive      0.786     0.781     0.783      1000

    accuracy                          0.784      2000
   macro avg      0.784     0.784     0.784      2000
weighted avg      0.784     0.784     0.784      2000

confusion matrix (rows = true, columns = predicted)
[[787 213]
 [219 781]]


In [25]:
name = "logreg"

model = build(name)
model.fit(x_train, y_train)
print(f"  features: {len(model.named_steps['tfidf'].get_feature_names_out())}")
print(f"  train accuracy: {model.score(x_train, y_train):.3f}")
print(f"  test  accuracy: {model.score(x_test, y_test):.3f}")
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"  5-fold CV: {scores.mean():.3f} ± {scores.std():.3f}  {scores.round(3)}")
predicted = evaluate(model, x_test, y_test, name)

  features: 9419
  train accuracy: 0.923
  test  accuracy: 0.780
  5-fold CV: 0.780 ± 0.006  [0.778 0.77  0.782 0.786 0.784]

--- logreg ---
              precision    recall  f1-score   support

    negative      0.769     0.801     0.785      1000
    positive      0.792     0.760     0.776      1000

    accuracy                          0.780      2000
   macro avg      0.781     0.780     0.780      2000
weighted avg      0.781     0.780     0.780      2000

confusion matrix (rows = true, columns = predicted)
[[801 199]
 [240 760]]


In [26]:
name = "svm"

model = build(name)
model.fit(x_train, y_train)
print(f"  features: {len(model.named_steps['tfidf'].get_feature_names_out())}")
print(f"  train accuracy: {model.score(x_train, y_train):.3f}")
print(f"  test  accuracy: {model.score(x_test, y_test):.3f}")
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"  5-fold CV: {scores.mean():.3f} ± {scores.std():.3f}  {scores.round(3)}")
predicted = evaluate(model, x_test, y_test, name)

  features: 9419
  train accuracy: 0.945
  test  accuracy: 0.762
  5-fold CV: 0.768 ± 0.005  [0.768 0.761 0.768 0.772 0.774]

--- svm ---
              precision    recall  f1-score   support

    negative      0.754     0.779     0.766      1000
    positive      0.771     0.746     0.759      1000

    accuracy                          0.762      2000
   macro avg      0.763     0.762     0.762      2000
weighted avg      0.763     0.762     0.762      2000

confusion matrix (rows = true, columns = predicted)
[[779 221]
 [254 746]]


In [28]:
name = "tree"

model = build(name)
model.fit(x_train, y_train)
print(f"  features: {len(model.named_steps['tfidf'].get_feature_names_out())}")
print(f"  train accuracy: {model.score(x_train, y_train):.3f}")
print(f"  test  accuracy: {model.score(x_test, y_test):.3f}")
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"  5-fold CV: {scores.mean():.3f} ± {scores.std():.3f}  {scores.round(3)}")
predicted = evaluate(model, x_test, y_test, name)

  features: 9419
  train accuracy: 0.878
  test  accuracy: 0.750
  5-fold CV: 0.732 ± 0.012  [0.749 0.716 0.732 0.721 0.742]

--- tree ---
              precision    recall  f1-score   support

    negative      0.717     0.825     0.767      1000
    positive      0.794     0.674     0.729      1000

    accuracy                          0.750      2000
   macro avg      0.755     0.750     0.748      2000
weighted avg      0.755     0.750     0.748      2000

confusion matrix (rows = true, columns = predicted)
[[825 175]
 [326 674]]


In [42]:
name = "forest"

model = build(name)
model.fit(x_train, y_train)
print(f"  features: {len(model.named_steps['tfidf'].get_feature_names_out())}")
print(f"  train accuracy: {model.score(x_train, y_train):.3f}")
print(f"  test  accuracy: {model.score(x_test, y_test):.3f}")
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"  5-fold CV: {scores.mean():.3f} ± {scores.std():.3f}  {scores.round(3)}")
predicted = evaluate(model, x_test, y_test, name)

  features: 9419
  train accuracy: 0.980
  test  accuracy: 0.775
  5-fold CV: 0.777 ± 0.011  [0.799 0.768 0.773 0.771 0.778]

--- forest ---
              precision    recall  f1-score   support

    negative      0.769     0.788     0.778      1000
    positive      0.783     0.763     0.773      1000

    accuracy                          0.775      2000
   macro avg      0.776     0.776     0.775      2000
weighted avg      0.776     0.775     0.775      2000

confusion matrix (rows = true, columns = predicted)
[[788 212]
 [237 763]]


### Ablation on Train size

In [33]:
explore(df_train, "train")


train: 134368 reviews
  label 0 (negative):  23951   17.8%
  label 1 (positive): 110417   82.2%
  domains: {'appstore': 101477, 'market': 22561, 'mapping': 6509, 'bookstore': 3821}
  words per review: median=6  mean=8.7  max=271
  very short (<= 3 words): 22.8%
  label 0 (negative): «Реклама коп»
  label 1 (positive): «Өтте күшті»


In [34]:
explore(df_test, "test")


test: 16797 reviews
  label 0 (negative):   2993   17.8%
  label 1 (positive):  13804   82.2%
  domains: {'appstore': 12685, 'market': 2820, 'mapping': 814, 'bookstore': 478}
  words per review: median=6  mean=8.6  max=137
  very short (<= 3 words): 22.5%
  label 0 (negative): «Рахатты гдзыдан аламыз»
  label 1 (positive): «тез тарту керек 5 минутта тартылатын болсын.»


In [31]:
def ablation(train_size, test_size):
    TRAIN_SIZE = train_size
    TEST_SIZE = test_size

    train = balance(df_train, TRAIN_SIZE)
    test = balance(df_test, TEST_SIZE)

    x_train, y_train = train["text_cleaned"].astype(str), train["label"]
    x_test, y_test = test["text_cleaned"].astype(str), test["label"]

    name = "nb"

    model = build(name)
    model.fit(x_train, y_train)
    print(f"  features: {len(model.named_steps['tfidf'].get_feature_names_out())}")
    print(f"  train accuracy: {model.score(x_train, y_train):.3f}")
    print(f"  test  accuracy: {model.score(x_test, y_test):.3f}")
    scores = cross_val_score(model, x_train, y_train, cv=5)
    print(f"  5-fold CV: {scores.mean():.3f} ± {scores.std():.3f}  {scores.round(3)}")
    predicted = evaluate(model, x_test, y_test, name)

In [32]:
ablation(4000, 1000)

  features: 9419
  train accuracy: 0.867
  test  accuracy: 0.784
  5-fold CV: 0.792 ± 0.006  [0.789 0.788 0.785 0.797 0.801]

--- nb ---
              precision    recall  f1-score   support

    negative      0.782     0.787     0.785      1000
    positive      0.786     0.781     0.783      1000

    accuracy                          0.784      2000
   macro avg      0.784     0.784     0.784      2000
weighted avg      0.784     0.784     0.784      2000

confusion matrix (rows = true, columns = predicted)
[[787 213]
 [219 781]]


In [35]:
ablation(8000, 2000)

  features: 17908
  train accuracy: 0.867
  test  accuracy: 0.797
  5-fold CV: 0.798 ± 0.003  [0.801 0.794 0.799 0.796 0.799]

--- nb ---
              precision    recall  f1-score   support

    negative      0.799     0.794     0.797      2000
    positive      0.796     0.800     0.798      2000

    accuracy                          0.797      4000
   macro avg      0.797     0.797     0.797      4000
weighted avg      0.797     0.797     0.797      4000

confusion matrix (rows = true, columns = predicted)
[[1589  411]
 [ 400 1600]]


In [37]:
ablation(16000, 2500)

  features: 34145
  train accuracy: 0.862
  test  accuracy: 0.800
  5-fold CV: 0.805 ± 0.001  [0.805 0.804 0.806 0.803 0.807]

--- nb ---
              precision    recall  f1-score   support

    negative      0.800     0.799     0.800      2500
    positive      0.800     0.801     0.800      2500

    accuracy                          0.800      5000
   macro avg      0.800     0.800     0.800      5000
weighted avg      0.800     0.800     0.800      5000

confusion matrix (rows = true, columns = predicted)
[[1998  502]
 [ 498 2002]]


In [39]:
ablation(2000, 500)

  features: 4846
  train accuracy: 0.875
  test  accuracy: 0.775
  5-fold CV: 0.783 ± 0.013  [0.766 0.781 0.776 0.784 0.806]

--- nb ---
              precision    recall  f1-score   support

    negative      0.777     0.772     0.774       500
    positive      0.773     0.778     0.776       500

    accuracy                          0.775      1000
   macro avg      0.775     0.775     0.775      1000
weighted avg      0.775     0.775     0.775      1000

confusion matrix (rows = true, columns = predicted)
[[386 114]
 [111 389]]


In [40]:
ablation(1000, 250)

  features: 2438
  train accuracy: 0.874
  test  accuracy: 0.778
  5-fold CV: 0.773 ± 0.014  [0.752 0.76  0.782 0.79  0.78 ]

--- nb ---
              precision    recall  f1-score   support

    negative      0.781     0.772     0.777       250
    positive      0.775     0.784     0.779       250

    accuracy                          0.778       500
   macro avg      0.778     0.778     0.778       500
weighted avg      0.778     0.778     0.778       500

confusion matrix (rows = true, columns = predicted)
[[193  57]
 [ 54 196]]


### Quality check

In [41]:
def top_words(model, n=10):
    """The features pushing hardest towards each class.

    Logistic Regression and LinearSVC expose one weight per feature in coef_.
    MultinomialNB has no weights — it stores log P(word | class), so the
    difference between the two rows plays the same role. Trees only report
    feature_importances_, which has no direction, so they return nothing here.
    """
    names = model.named_steps["tfidf"].get_feature_names_out()
    classifier = model.named_steps["clf"]
    if hasattr(classifier, "coef_"):                     # LogisticRegression, LinearSVC
        weights = classifier.coef_[0]
    elif hasattr(classifier, "feature_log_prob_"):       # MultinomialNB
        weights = classifier.feature_log_prob_[1] - classifier.feature_log_prob_[0]
    else:                                                # trees: importance has no sign
        return [], []
    order = weights.argsort()
    negative = [names[i] for i in order if names[i] not in SKIP][:n]
    positive = [names[i] for i in order[::-1] if names[i] not in SKIP][:n]
    return negative, positive


def show_mistakes(x_test, y_test, predicted, limit=4):
    wrong = [(t, y, p) for t, y, p in zip(x_test, y_test, predicted) if y != p]
    for text, true, got in wrong[:limit]:
        print(f"  true={LABELS[true]:>8}  predicted={LABELS[got]:>8}  «{str(text)[:55]}»")

In [45]:
TRAIN_SIZE = 4000
TEST_SIZE = 1000

train = balance(df_train, TRAIN_SIZE)
test = balance(df_test, TEST_SIZE)

x_train, y_train = train["text_cleaned"].astype(str), train["label"]
x_test, y_test = test["text_cleaned"].astype(str), test["label"]

In [48]:
name = "nb"

model = build(name)
model.fit(x_train, y_train)
print(f"  features: {len(model.named_steps['tfidf'].get_feature_names_out())}")
print(f"  train accuracy: {model.score(x_train, y_train):.3f}")
print(f"  test  accuracy: {model.score(x_test, y_test):.3f}")
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"  5-fold CV: {scores.mean():.3f} ± {scores.std():.3f}  {scores.round(3)}")
predicted = evaluate(model, x_test, y_test, name)

  features: 9419
  train accuracy: 0.867
  test  accuracy: 0.784
  5-fold CV: 0.792 ± 0.006  [0.789 0.788 0.785 0.797 0.801]

--- nb ---
              precision    recall  f1-score   support

    negative      0.782     0.787     0.785      1000
    positive      0.786     0.781     0.783      1000

    accuracy                          0.784      2000
   macro avg      0.784     0.784     0.784      2000
weighted avg      0.784     0.784     0.784      2000

confusion matrix (rows = true, columns = predicted)
[[787 213]
 [219 781]]


In [51]:
best = (model, predicted)

SKIP = {}
negative, positive = top_words(model, n=30)
if negative:
    print("  words pointing to negative:", ", ".join(negative))
    print("  words pointing to positive:", ", ".join(positive))

  words pointing to negative: нашар, өте нашар, өте жаман, ұнамады, мүлдем, жасамайды, маған ұнамады, оте нашар, түсініксіз, қате, фуу, жазылмайды, жаман, жаман ойын, дым, тартылмайды, ешак, мал, шығып, не, түк, алмайсын, загрузка, ашпайды, пароль, дурыс, смс, ештене, зайбал, істемейді
  words pointing to positive: keremet, разы, алла, разы болсын, аллаһ, кеңес беремін, алла разы, рахмет, ыңғайлы, өте ыңғайлы, жеткізді, көп рахмет, өте керемет, жақсы екен, жеңіл, берді, бізге ұнады, уа, уақытынан, сапасы жақсы, сапалы, өте жақсы, аллаһ разы, рақмет, сіздерге, керемет қосымша, жаусын, әдемі, құрал, рахмет сіздерге


In [52]:
print("\n  mistakes:")
show_mistakes(x_test, y_test, predicted)


  mistakes:
  true=positive  predicted=negative  «мен қазақшамың қазақстанда тұрамын»
  true=positive  predicted=negative  «ма шааллагь»
  true=negative  predicted=positive  «бауырларым менде жүмғой»
  true=positive  predicted=negative  «жаңа әндер мен өлеңдер жинағы мен сені чаттан бұл кезде»
